# Assignment 3 – ACID Transactions on a B+ Tree Database

This notebook demonstrates full ACID transaction support built on top of the existing B+ Tree storage engine from Assignment 2.

### Three relations used:
| Table | Primary Key | Fields |
|-------|-------------|--------|
| `users` | `user_id` | user_id, name, balance, city |
| `products` | `product_id` | product_id, name, stock, price |
| `orders` | `order_id` | order_id, user_id, product_id, amount, timestamp |

## 0. Setup – create database, tables, and seed data

In [2]:
from db_manager import DatabaseManager
from transaction_manager import TransactionManager
import time
import threading

# ── bootstrap ─────────────────────────────────────────────────────────
db = DatabaseManager()
db.create_database('shop_db')

db.create_table('shop_db', 'users', {
    'user_id': int, 'name': str, 'balance': float, 'city': str
}, order=8, search_key='user_id')

db.create_table('shop_db', 'products', {
    'product_id': int, 'name': str, 'stock': int, 'price': float
}, order=8, search_key='product_id')

db.create_table('shop_db', 'orders', {
    'order_id': int, 'user_id': int, 'product_id': int,
    'amount': float, 'timestamp': float
}, order=8, search_key='order_id')

# ── seed data ─────────────────────────────────────────────────────────
users_tbl,    _ = db.get_table('shop_db', 'users')
products_tbl, _ = db.get_table('shop_db', 'products')
orders_tbl,   _ = db.get_table('shop_db', 'orders')

users_tbl.insert({'user_id': 1, 'name': 'Alice',   'balance': 500.0,  'city': 'Delhi'})
users_tbl.insert({'user_id': 2, 'name': 'Bob',     'balance': 300.0,  'city': 'Mumbai'})
users_tbl.insert({'user_id': 3, 'name': 'Charlie', 'balance': 1000.0, 'city': 'Chennai'})

products_tbl.insert({'product_id': 101, 'name': 'Laptop',  'stock': 10, 'price': 800.0})
products_tbl.insert({'product_id': 102, 'name': 'Phone',   'stock': 25, 'price': 400.0})
products_tbl.insert({'product_id': 103, 'name': 'Headset', 'stock': 50, 'price': 80.0})

# Transaction Manager (creates WAL)
tm = TransactionManager(db, wal_path='wal.log')

def show(label, tbl):
    print(f'\n── {label} ─────────────────────────')
    for _, v in tbl.get_all():
        print(' ', v)

show('users',    users_tbl)
show('products', products_tbl)
print('\nSetup complete.')


── users ─────────────────────────
  {'user_id': 1, 'name': 'Alice', 'balance': 500.0, 'city': 'Delhi'}
  {'user_id': 2, 'name': 'Bob', 'balance': 300.0, 'city': 'Mumbai'}
  {'user_id': 3, 'name': 'Charlie', 'balance': 1000.0, 'city': 'Chennai'}

── products ─────────────────────────
  {'product_id': 101, 'name': 'Laptop', 'stock': 10, 'price': 800.0}
  {'product_id': 102, 'name': 'Phone', 'stock': 25, 'price': 400.0}
  {'product_id': 103, 'name': 'Headset', 'stock': 50, 'price': 80.0}

Setup complete.


---
## 1. Atomicity – multi-table COMMIT

A single transaction:
1. Deducts Alice's balance (users)
2. Decrements Laptop stock (products)
3. Inserts a new order (orders)

All three succeed → COMMIT

In [3]:
tx = tm.begin()

# Step 1: deduct Alice's balance
alice = users_tbl.get(1)
ok, msg = tm.update(tx, 'shop_db', 'users', 1,
    {**alice, 'balance': alice['balance'] - 800.0})
print('Update user:', ok, msg)

# Step 2: decrement Laptop stock
laptop = products_tbl.get(101)
ok, msg = tm.update(tx, 'shop_db', 'products', 101,
    {**laptop, 'stock': laptop['stock'] - 1})
print('Update product:', ok, msg)

# Step 3: insert order
ok, msg = tm.insert(tx, 'shop_db', 'orders', {
    'order_id': 1001, 'user_id': 1, 'product_id': 101,
    'amount': 800.0, 'timestamp': time.time()
})
print('Insert order:', ok, msg)

ok, msg = tm.commit(tx)
print('COMMIT:', ok, msg)

show('users after commit',    users_tbl)
show('products after commit', products_tbl)
show('orders after commit',   orders_tbl)

[TxManager] BEGIN  tx=87fbba09
Update user: True Updated key=1 in users
Update product: True Updated key=101 in products
Insert order: True Inserted key=1001 into orders
[TxManager] COMMIT tx=87fbba09
COMMIT: True Transaction 87fbba09 committed successfully

── users after commit ─────────────────────────
  {'user_id': 1, 'name': 'Alice', 'balance': -300.0, 'city': 'Delhi'}
  {'user_id': 2, 'name': 'Bob', 'balance': 300.0, 'city': 'Mumbai'}
  {'user_id': 3, 'name': 'Charlie', 'balance': 1000.0, 'city': 'Chennai'}

── products after commit ─────────────────────────
  {'product_id': 101, 'name': 'Laptop', 'stock': 9, 'price': 800.0}
  {'product_id': 102, 'name': 'Phone', 'stock': 25, 'price': 400.0}
  {'product_id': 103, 'name': 'Headset', 'stock': 50, 'price': 80.0}

── orders after commit ─────────────────────────
  {'order_id': 1001, 'user_id': 1, 'product_id': 101, 'amount': 800.0, 'timestamp': 1774731764.7679298}


---
## 2. Atomicity – Crash / Failure mid-transaction → ROLLBACK

We start a new transaction, apply two operations, then **simulate a failure** before the third.
ROLLBACK must undo all applied changes — leaving the database in its pre-transaction state.

In [5]:
print('=== State BEFORE transaction ===')
print('Bob balance:', users_tbl.get(2)['balance'])
print('Phone stock:', products_tbl.get(102)['stock'])
print('Orders:', [k for k, _ in orders_tbl.get_all()])

tx2 = tm.begin()

# Op 1: deduct Bob's balance
bob = users_tbl.get(2)
tm.update(tx2, 'shop_db', 'users', 2,
    {**bob, 'balance': bob['balance'] - 400.0})

# Op 2: decrement Phone stock
phone = products_tbl.get(102)
tm.update(tx2, 'shop_db', 'products', 102,
    {**phone, 'stock': phone['stock'] - 1})

print('=== State BEFORE failing transaction ===')
print('Bob balance:', users_tbl.get(2)['balance'])
print('Phone stock:', products_tbl.get(102)['stock'])
print('Orders:', [k for k, _ in orders_tbl.get_all()])


# ── SIMULATED FAILURE ─────────────────────────────────────────
print('\n*** SIMULATED CRASH before inserting order ***')
ok, msg = tm.rollback(tx2)
print('ROLLBACK result:', ok, msg)
# ──────────────────────────────────────────────────────────────

print('\n=== State AFTER rollback (must match BEFORE) ===')
print('Bob balance:', users_tbl.get(2)['balance'])      # must be 300.0
print('Phone stock:', products_tbl.get(102)['stock'])   # must be 25
print('Orders:', [k for k, _ in orders_tbl.get_all()])

=== State BEFORE transaction ===
Bob balance: 300.0
Phone stock: 25
Orders: [1001]
[TxManager] BEGIN  tx=a5ec1e3b
=== State BEFORE failing transaction ===
Bob balance: -100.0
Phone stock: 24
Orders: [1001]

*** SIMULATED CRASH before inserting order ***
[TxManager] ABORT  tx=a5ec1e3b
ROLLBACK result: True Transaction a5ec1e3b rolled back successfully

=== State AFTER rollback (must match BEFORE) ===
Bob balance: 300.0
Phone stock: 25
Orders: [1001]


---
## 3. Consistency – constraint enforcement

We try to set a balance below 0 (negative balance is invalid).  
The constraint is checked explicitly before commit; on violation we ROLLBACK.

In [6]:
def check_balance_positive(users_table):
    for _, record in users_table.get_all():
        if record['balance'] < 0:
            return False, f"User {record['user_id']} has negative balance {record['balance']}"
    return True, 'OK'

tx3 = tm.begin()

charlie = users_tbl.get(3)
# Try to overdraft Charlie (balance 1000, spending 2000)
tm.update(tx3, 'shop_db', 'users', 3,
    {**charlie, 'balance': charlie['balance'] - 2000.0})

# Consistency check
valid, reason = check_balance_positive(users_tbl)
if not valid:
    print(f'Constraint violated: {reason}')
    tm.rollback(tx3)
    print('Transaction rolled back for consistency.')
else:
    tm.commit(tx3)

print('Charlie balance after check:', users_tbl.get(3)['balance'])  # must be 1000.0

[TxManager] BEGIN  tx=d74cc949
Constraint violated: User 1 has negative balance -300.0
[TxManager] ABORT  tx=d74cc949
Transaction rolled back for consistency.
Charlie balance after check: 1000.0


---
## 4. Isolation – concurrent transactions on the same table

Two threads try to modify the **same table** simultaneously.  
The `LockManager` serialises them — the second thread waits for the first to commit/rollback.
No intermediate state is visible and no corruption occurs.

In [7]:
results = []

def thread_a():
    tx = tm.begin()
    headset = products_tbl.get(103)
    tm.update(tx, 'shop_db', 'products', 103,
        {**headset, 'stock': headset['stock'] - 5})
    time.sleep(0.2)   # hold lock briefly to force conflict
    ok, msg = tm.commit(tx)
    results.append(('A', ok, msg))

def thread_b():
    time.sleep(0.05)  # start slightly after A
    tx = tm.begin()
    try:
        headset = products_tbl.get(103)
        tm.update(tx, 'shop_db', 'products', 103,
            {**headset, 'stock': headset['stock'] - 3})
        ok, msg = tm.commit(tx)
        results.append(('B', ok, msg))
    except Exception as e:
        results.append(('B', False, str(e)))
        tm.rollback(tx)

t1 = threading.Thread(target=thread_a)
t2 = threading.Thread(target=thread_b)
t1.start(); t2.start()
t1.join(); t2.join()

print('Thread results:')
for r in results:
    print(f'  Thread {r[0]}: success={r[1]}, msg={r[2]}')

final_stock = products_tbl.get(103)['stock']
print(f'\nHeadset stock: started=50, expected=42, actual={final_stock}')
assert final_stock == 42, 'Isolation violation!'
print('Isolation verified ✓')

[TxManager] BEGIN  tx=b8d302a5
[TxManager] BEGIN  tx=1db1d5f4
[TxManager] COMMIT tx=b8d302a5
[TxManager] COMMIT tx=1db1d5f4
Thread results:
  Thread A: success=True, msg=Transaction b8d302a5 committed successfully
  Thread B: success=True, msg=Transaction 1db1d5f4 committed successfully

Headset stock: started=50, expected=42, actual=42
Isolation verified ✓


---
## 5. Durability – persist committed data across restart

We commit a transaction, then **simulate a restart** by re-creating the `DatabaseManager` and `TransactionManager` from scratch, re-populating from stored data, and verifying the committed record is still present.

In [8]:
import pickle, os

PERSIST_FILE = 'db_snapshot.pkl'

def save_db(db, path=PERSIST_FILE):
    snapshot = {}
    for db_name, tables in db.databases.items():
        snapshot[db_name] = {}
        for tname, table in tables.items():
            snapshot[db_name][tname] = {
                'schema': table.schema,
                'order': table.order,
                'search_key': table.search_key,
                'records': table.get_all(),
            }
    with open(path, 'wb') as f:
        pickle.dump(snapshot, f)
    print(f'Snapshot saved to {path}')

def load_db(path=PERSIST_FILE):
    from db_manager import DatabaseManager
    new_db = DatabaseManager()
    with open(path, 'rb') as f:
        snapshot = pickle.load(f)
    for db_name, tables in snapshot.items():
        new_db.create_database(db_name)
        for tname, info in tables.items():
            new_db.create_table(db_name, tname, info['schema'],
                                order=info['order'],
                                search_key=info['search_key'])
            tbl, _ = new_db.get_table(db_name, tname)
            for _, record in info['records']:
                tbl.insert(record)
    print(f'Database restored from {path}')
    return new_db

# ── commit a new record ───────────────────────────────────────────────
tx_dur = tm.begin()
tm.insert(tx_dur, 'shop_db', 'orders', {
    'order_id': 9999, 'user_id': 3, 'product_id': 103,
    'amount': 80.0, 'timestamp': time.time()
})
tm.commit(tx_dur)
print('Order 9999 committed.')

# ── save & reload ─────────────────────────────────────────────────────
save_db(db)
restored_db = load_db()
restored_tm = TransactionManager(restored_db, wal_path='wal_restored.log')

restored_orders, _ = restored_db.get_table('shop_db', 'orders')
record = restored_orders.get(9999)
print(f'\nOrder 9999 after restart: {record}')
assert record is not None and record['order_id'] == 9999
print('Durability verified ✓')

[TxManager] BEGIN  tx=6489ca82
[TxManager] COMMIT tx=6489ca82
Order 9999 committed.
Snapshot saved to db_snapshot.pkl
Database restored from db_snapshot.pkl

Order 9999 after restart: {'order_id': 9999, 'user_id': 3, 'product_id': 103, 'amount': 80.0, 'timestamp': 1774731983.2740724}
Durability verified ✓


---
## 6. WAL Recovery – undo incomplete transaction after crash

We write BEGIN + two ops to the WAL but **never write COMMIT**.  
On restart, `TransactionManager.recover()` detects the incomplete entry and rolls it back automatically.

In [9]:
import json, time as _time

# Inject a 'crashed' transaction directly into the WAL file
crash_tx_id = 'crash001'
with open('wal_crash.log', 'w') as f:
    f.write(json.dumps({'type': 'BEGIN',  'tx_id': crash_tx_id, 'timestamp': _time.time()}) + '\n')
    f.write(json.dumps({
        'type': 'OP', 'tx_id': crash_tx_id, 'op': 'INSERT',
        'table': 'orders',
        'key': 7777,
        'before': None,
        'after': {'order_id': 7777, 'user_id': 2, 'product_id': 101,
                  'amount': 999.0, 'timestamp': _time.time()},
        'timestamp': _time.time()
    }) + '\n')
    # No COMMIT → simulates crash

# Apply the orphan insert manually so recovery has something to undo
orders_tbl.insert({'order_id': 7777, 'user_id': 2, 'product_id': 101,
                   'amount': 999.0, 'timestamp': _time.time()})
print('Orphan order 7777 inserted (simulating crash mid-tx).')
print('Orders before recovery:', [k for k, _ in orders_tbl.get_all()])

# Recovery: new TransactionManager reads WAL and undoes incomplete tx
tm_rec = TransactionManager(db, wal_path='wal_crash.log')

print('Orders after recovery:', [k for k, _ in orders_tbl.get_all()])
assert orders_tbl.get(7777) is None, 'Recovery failed – orphan order still present!'
print('WAL recovery verified ✓')

Orphan order 7777 inserted (simulating crash mid-tx).
Orders before recovery: [1001, 7777, 9999]
[Recovery] Rolling back incomplete tx=crash001 (1 ops)
[Recovery] Rolled back 1 incomplete transaction(s)
[Recovery] WAL compacted
Orders after recovery: [1001, 9999]
WAL recovery verified ✓
